# 03.2 Indentation Rules and Blocks

In almost every other language, indentation is decoration — the compiler ignores
it, and it exists purely so humans can read the code. In Python it is **syntax**.
The indentation *is* the structure.

This is Python's most distinctive design decision, and the source of the errors
beginners hit most often.

## Theory

### What a block is

A **block** is a group of statements that belong together. Most languages mark
blocks with braces:

```c
if (x > 0) {
    printf("positive");
    printf("done");
}
```

Python marks them with a colon and indentation:

```python
if x > 0:
    print("positive")
    print("done")
```

The rule is exact:

1. A statement ending in `:` **opens a block**
2. Every line indented further belongs to that block
3. The block ends when indentation returns to the previous level

### How the tokenizer actually sees it

This is the part worth understanding properly, because it explains every
indentation error you will ever get.

Before Python parses your code, a **tokenizer** breaks it into tokens. When it
reaches the start of a line, it compares that line's indentation with the
previous line's:

- **More indented?** Emit an `INDENT` token.
- **Less indented?** Emit one `DEDENT` token per level closed.
- **Same?** Emit nothing.

So Python genuinely does insert invisible "open brace" and "close brace" tokens.
They are just spelled as whitespace. You can see them directly, which we do
below.

### Why the indentation must match exactly

When a line dedents, its indentation must match **an enclosing level that already
exists**. If it lands between two levels, the tokenizer cannot tell which block
you meant to close, and raises `IndentationError`.

```python
if x:
    if y:
        a()
      b()      # matches neither 8 nor 4 - error
```

### Tabs versus spaces

A tab is one character but displays as several columns. Python 3 **refuses** to
guess how wide your tabs are, so mixing tabs and spaces in a way that makes the
indentation ambiguous raises `TabError`.

The rule is simply: **use four spaces, never tabs.** Configure your editor once
and the problem disappears permanently. This is PEP 8, and it is universal.

In [ ]:
import tokenize
import io

# A small piece of code with two levels of nesting.
source = """if True:
    x = 1
    if x:
        y = 2
z = 3
"""

print("SOURCE:")
for line_number, line in enumerate(source.splitlines(), start=1):
    # Show each line with its number so we can map tokens back to it.
    print(f"  {line_number} | {line}")

print("")
print("TOKENS the tokenizer produced (structure only):")
print("")

# tokenize needs a file-like object, so wrap the string in StringIO.
stream = io.StringIO(source)

for token in tokenize.generate_tokens(stream.readline):
    name = tokenize.tok_name[token.type]

    # Show only the tokens that carry block structure.
    if name in ("INDENT", "DEDENT", "NEWLINE", "NAME"):
        # DEDENT and NEWLINE have no visible text, so label them.
        text = repr(token.string) if token.string.strip() else ""
        print(f"   {name:8} {text}")

Look at the output above. `INDENT` and `DEDENT` are **real tokens**, generated
purely from whitespace. Python's grammar then uses them exactly the way C's
grammar uses `{` and `}`.

That is the whole mechanism. Indentation is not a style rule that Python happens
to enforce — it is how blocks are delimited.

## Every statement that opens a block

Any statement ending in `:` opens a block. Here is the complete set.

In [ ]:
# Each entry is (statement, a minimal example).
block_openers = [
    ("if / elif / else", "if x > 0:"),
    ("for", "for item in items:"),
    ("while", "while running:"),
    ("def", "def greet(name):"),
    ("class", "class Person:"),
    ("try / except / else / finally", "try:"),
    ("with", "with open(path) as f:"),
    ("match / case", "match command:"),
    ("async def", "async def fetch():"),
    ("async for / async with", "async with session:"),
]

print("Statement                        Opens a block like")
print("-" * 62)
for statement, example in block_openers:
    print(statement.ljust(33), example)

print("")
print("All of them share one rule: colon, newline, indent.")

## Nesting: blocks inside blocks

Each level of nesting adds four more spaces. There is no limit, but depth is a
warning sign — code nested more than three levels deep is usually asking to be
split into functions.

In [ ]:
# Four levels of nesting, to show how the levels stack.
orders = [
    {"id": 1, "paid": True, "items": ["book", "pen"]},
    {"id": 2, "paid": False, "items": ["laptop"]},
    {"id": 3, "paid": True, "items": []},
]

# Level 0: module level, no indentation.
for order in orders:
    # Level 1: inside the for loop - 4 spaces.
    print("order", order["id"])

    if order["paid"]:
        # Level 2: inside the if - 8 spaces.
        if order["items"]:
            # Level 3: inside the nested if - 12 spaces.
            for item in order["items"]:
                # Level 4: inside the inner for - 16 spaces.
                print("      shipping", item)
        else:
            print("      paid but nothing to ship")
    else:
        print("      awaiting payment")

### Flattening deep nesting

The code above works, but four levels is hard to follow. The usual fix is a
**guard clause** — handle the exceptional case first and move on, so the main
logic stays at a shallow level.

Compare these two versions of the same logic.

In [ ]:
def describe_order_nested(order):
    """Deeply nested version - the happy path is buried at the bottom."""
    if order["paid"]:
        if order["items"]:
            if len(order["items"]) > 1:
                return f"shipping {len(order['items'])} items"
            else:
                return f"shipping {order['items'][0]}"
        else:
            return "paid, nothing to ship"
    else:
        return "awaiting payment"


def describe_order_flat(order):
    """Guard-clause version - exceptions leave early, main logic stays flat."""
    # Deal with each exceptional case and return immediately.
    if not order["paid"]:
        return "awaiting payment"

    if not order["items"]:
        return "paid, nothing to ship"

    # By here, everything is normal. The main logic sits at one level.
    if len(order["items"]) > 1:
        return f"shipping {len(order['items'])} items"

    return f"shipping {order['items'][0]}"


# Both produce identical results.
test_orders = [
    {"paid": True, "items": ["book", "pen"]},
    {"paid": True, "items": ["laptop"]},
    {"paid": True, "items": []},
    {"paid": False, "items": ["phone"]},
]

print("Order state                    Nested            Flat")
print("-" * 68)
for order in test_orders:
    nested_result = describe_order_nested(order)
    flat_result = describe_order_flat(order)
    label = f"paid={order['paid']}, {len(order['items'])} items"
    # Confirm the two versions agree.
    match = "same" if nested_result == flat_result else "DIFFERENT"
    print(label.ljust(30), nested_result.ljust(17), match)

## The three indentation errors, and what each means

Python raises three distinct errors for indentation problems. Knowing which is
which cuts your debugging time enormously.

In [ ]:
# NEWLINE is built explicitly so these strings hold real line breaks
# without needing escape characters inside this cell.
NEWLINE = chr(10)

# Each entry is (label, the broken source, what went wrong).
broken = [
    (
        "expected an indented block",
        "if True:" + NEWLINE + "print('x')" + NEWLINE,
        "a colon opened a block, but the next line was not indented",
    ),
    (
        "unexpected indent",
        "x = 1" + NEWLINE + "    y = 2" + NEWLINE,
        "a line was indented when no block was open",
    ),
    (
        "unindent does not match",
        "if True:" + NEWLINE + "    x = 1" + NEWLINE + "  y = 2" + NEWLINE,
        "the dedent landed between two existing levels",
    ),
]

for label, source, explanation in broken:
    print("CASE:", label)

    # Show the offending source, one line at a time.
    for line in source.rstrip(NEWLINE).split(NEWLINE):
        print("   |" + line)

    try:
        compile(source, "<demo>", "exec")
        print("   compiled (unexpected)")
    except IndentationError as error:
        print("   IndentationError:", error.msg)
        print("   MEANING:", explanation)
    print("")

## `TabError`: why mixing tabs and spaces fails

Python 3 treats tabs and spaces as genuinely different. If a file indents some
lines with tabs and others with spaces, and the result is ambiguous, you get a
`TabError` — a subclass of `IndentationError`.

The demonstration below uses explicit `\t` characters so you can see exactly
what is happening.

In [ ]:
# Build the sources explicitly so the tabs are unambiguous in this file.
NEWLINE = chr(10)
TAB = chr(9)
FOUR_SPACES = "    "

# CASE 1: a tab and four spaces used at what looks like the same level.
same_level = "if True:" + NEWLINE + TAB + "x = 1" + NEWLINE + FOUR_SPACES + "y = 2" + NEWLINE

# CASE 2: spaces for the outer block, a tab for the nested one.
nested = ("if True:" + NEWLINE
          + FOUR_SPACES + "if True:" + NEWLINE
          + TAB + "x = 1" + NEWLINE)

for label, source in [("tab and spaces at one level", same_level),
                      ("spaces outside, tab inside", nested)]:
    print("CASE:", label)

    # repr() makes the tab visible as a backslash-t escape.
    for line_number, line in enumerate(source.rstrip(NEWLINE).split(NEWLINE), start=1):
        print("   ", line_number, "|", repr(line))

    try:
        compile(source, "<demo>", "exec")
        print("    compiled (unexpected)")
    except TabError as error:
        # TabError is raised when the mix is genuinely ambiguous.
        print("    TabError:", error.msg)
    except IndentationError as error:
        # Otherwise Python reports the simpler indentation problem.
        print("    IndentationError:", error.msg)
    print("")

print("Both cases are caused by the same mistake: mixing tabs and spaces.")
print("Python reports whichever problem it detects first, which is why")
print("you may see either error from the same root cause.")
print("")
print("THE FIX: use four spaces everywhere. Set your editor to insert")
print("spaces when you press Tab, and both errors become impossible.")

## Indentation size: what is actually allowed

Python requires **consistency**, not a specific number. Any amount works as long
as a block uses the same amount throughout.

But PEP 8 says four spaces, and effectively all Python code follows it. The
examples below prove the flexibility — then you should ignore it and use four.

In [ ]:
NEWLINE = chr(10)

# Several indentation widths, each valid on its own.
variants = [
    ("1 space", "if True:" + NEWLINE + " x = 1" + NEWLINE),
    ("2 spaces", "if True:" + NEWLINE + "  x = 1" + NEWLINE),
    ("4 spaces", "if True:" + NEWLINE + "    x = 1" + NEWLINE),
    ("8 spaces", "if True:" + NEWLINE + "        x = 1" + NEWLINE),
]

print("Width       Compiles?")
print("-" * 26)
for label, source in variants:
    try:
        compile(source, "<demo>", "exec")
        print(label.ljust(12), "yes")
    except IndentationError:
        print(label.ljust(12), "no")

print("")
print("Python requires CONSISTENCY, not a specific width.")
print("But PEP 8 says four spaces, and every codebase follows it.")

## Where indentation does *not* apply

Inside brackets — `()`, `[]`, `{}` — Python ignores line breaks and indentation
entirely. This is called **implicit line joining**, and it is how long
expressions get formatted readably.

Covered fully in 03.4.

In [ ]:
# Inside brackets, indentation is free - Python only looks for the closing bracket.
config = {
        "host": "localhost",
    "port": 8080,
            "debug": True,
}

print("The dict parsed fine despite ragged indentation:")
print("  ", config)

# The same applies to function calls and list literals.
numbers = [
    1, 2, 3,
        4, 5, 6,
]
print("")
print("List parsed fine too:", numbers)

print("")
print("Readable style keeps one item per line, indented consistently:")

readable_config = {
    "host": "localhost",
    "port": 8080,
    "debug": True,
}
for key, value in readable_config.items():
    print(f"   {key:<8} {value}")

## `pass`: the empty block

Python has no way to write an empty block — the indentation has nothing to
delimit. `pass` is the placeholder that solves this: a statement that does
nothing, purely so a block can exist.

In [ ]:
# A block cannot be empty, so this fails.
try:
    compile("if True:" + chr(10), "<demo>", "exec")
except IndentationError as error:
    print("Empty block:", error.msg)

# pass fills the gap.
if True:
    pass

print("")
print("pass compiled fine - it is a statement that does nothing.")

# Real uses for pass:
uses = [
    ("Stub a function you will write later", "def process(): pass"),
    ("Define an exception class with no body", "class MyError(Exception): pass"),
    ("Deliberately ignore one case", "except KeyError: pass"),
    ("Placeholder while sketching structure", "for x in items: pass"),
]

print("")
print("When pass is used:")
for purpose, example in uses:
    print("   " + purpose)
    print("      " + example)

print("")
print("Note: `except SomeError: pass` silently swallows an error.")
print("The Zen of Python warns against this. See Chapter 23.")

## Takeaways

1. Indentation is **syntax** in Python, not style. It delimits blocks.
2. The tokenizer emits real `INDENT` and `DEDENT` tokens from whitespace —
   they are Python's `{` and `}`.
3. A statement ending in `:` opens a block; the block ends when indentation
   returns to the enclosing level.
4. A dedent must match a level that already exists, or you get
   `IndentationError: unindent does not match`.
5. Never mix tabs and spaces — Python 3 raises `TabError` rather than guess.
6. Use **four spaces** per level. PEP 8, and universal in practice.
7. Inside brackets, indentation is ignored entirely.
8. `pass` exists because a block cannot be empty.
9. Deep nesting is a smell — flatten it with guard clauses.

## Try it yourself

1. Run the tokenizer example on your own nested code. Count the `INDENT` and
   `DEDENT` tokens — do they balance?
2. Write a four-level nested `if`, then rewrite it with guard clauses. Which
   version reads better?
3. Deliberately produce each of the three indentation errors and read the
   messages until they are familiar.
4. In your editor, turn on "render whitespace". Can you now see the difference
   between a tab and four spaces?